In [ ]:
import sys
import os

sys.path.append(os.path.abspath(".."))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yfinance as yf
from src.backtest import run_momentum_backtest

from src.metrics import calculate_performance_metrics
from src.factors import calculate_momentum, calculate_volatility, zscore_factor


In [8]:
prices = pd.read_csv(
    "../data/prices.csv",
    index_col=0,
    parse_dates=True
)

prices.head()

momentum = calculate_momentum(prices, lookback=252)

latest_momentum = momentum.iloc[-1]

In [9]:
#Compute returns
returns = prices.pct_change()

#Compute volatility
volatility = calculate_volatility(returns)

#Get latest volatility
latest_vol = volatility.iloc[-1]


#Sort latest volatility
latest_vol.sort_values()

#Compute z-score of latest volatility
momentum_z = zscore_factor(latest_momentum)

low_vol_z = zscore_factor(-latest_vol)

#Combine the z-scores of momentum and low volatility to create a combined factor score
combined_score = (
    0.5 * momentum_z
    + 0.5 * low_vol_z
)

In [10]:
#Create a DataFrame to hold the multi-factor scores
multi_factor_table = pd.DataFrame({
    "momentum": latest_momentum,
    "momentum_z": momentum_z,
    "volatility": latest_vol,
    "low_vol_z": low_vol_z,
    "combined_score": combined_score
})

multi_factor_table = multi_factor_table.sort_values(
    "combined_score",
    ascending=False
)

multi_factor_table.head(10)

,momentum,momentum_z,volatility,low_vol_z,combined_score
WMT,0.739923,1.358378,0.011142,0.792457,1.075418
KO,0.088775,-0.409008,0.008059,1.410240,0.500616
AXP,0.603201,0.987280,0.015187,-0.018391,0.484444
PG,0.172538,-0.181654,0.009442,1.133190,0.475768
V,0.223183,-0.044190,0.010636,0.893806,0.424808
CSCO,0.209956,-0.080091,0.011352,0.750330,0.335119
JPM,0.442881,0.552129,0.014800,0.059348,0.305738
GS,0.520313,0.762300,0.016226,-0.226470,0.267915
IBM,0.392704,0.415936,0.014739,0.071482,0.243709
NVDA,1.712493,3.998187,0.033074,-3.603278,0.197455


In [11]:
#Get the top 5 stocks based on the combined factor score
top5_multi_factor = multi_factor_table.head(5).index

top5_multi_factor

Index(['WMT', 'KO', 'AXP', 'PG', 'V'], dtype='str')

In [13]:
from src.backtest import run_momentum_low_vol_backtest

from src.metrics import calculate_growth, calculate_performance_metrics
# the multi metrics for comparision
multi_portfolio, multi_holdings = run_momentum_low_vol_backtest(

    prices,

    lookback=252,

    vol_window=252,

    top_n=5,

    momentum_weight=0.5,

    low_vol_weight=0.5

)

multi_growth = calculate_growth(multi_portfolio)

multi_metrics = calculate_performance_metrics(multi_portfolio)

multi_metrics

CAGR                 0.137556
Annual Return        0.143997
Annual Volatility    0.182588
Sharpe Ratio         0.788642
Max Drawdown        -0.164117
dtype: float64

In [14]:
# Recreate the momentum metrics for comparision
portfolio, holdings_history = run_momentum_backtest(
    prices,
    lookback=252,
    top_n=5,
)
momentum_metrics = calculate_performance_metrics(portfolio)

# Recreate the spy metrics for comparision
spy = yf.download(
    "SPY",
    start="2018-01-01",
    end="2025-01-01",
    auto_adjust=True,
)

spy_close = spy["Close"]["SPY"] if isinstance(spy["Close"], pd.DataFrame) else spy["Close"]
spy_returns = spy_close.resample("ME").last().pct_change().loc[portfolio.index]
spy_metrics = calculate_performance_metrics(spy_returns)

comparison_phase2 = pd.DataFrame({
    "Momentum": momentum_metrics,
    "Momentum + Low Vol": multi_metrics,
    "SPY": spy_metrics
})

comparison_phase2

[*********************100%***********************]  1 of 1 completed


,Momentum,Momentum + Low Vol,SPY
CAGR,0.236374,0.137556,0.160590
Annual Return,0.231468,0.143997,0.162722
Annual Volatility,0.205005,0.182588,0.173038
Sharpe Ratio,1.129083,0.788642,0.940383
Max Drawdown,-0.212236,-0.164117,-0.239272


Adding a low-volatility factor reduced risk, but also reduced returns enough that overall performance declined.